# 최종 목표
문맥을 이해하고 빈칸을 예측하는 모델

`나는 오늘 [MASK] 를 먹었다.` 에서 `[MASK]`를 예측하는 모델

참고하는 저장소: https://github.com/shreydan/masked-language-modeling

In [35]:
import math
import torch
from torch import nn

B, T, D = 1, 4, 8

num_heads = 2
head_dim = D // num_heads

torch.manual_seed(42)

In [36]:
# 1. 토큰 4개를 가진 하나의 문장
token_ids = torch.tensor([[3, 7, 2, 9]])
token_ids.dtype

torch.int64

In [37]:
# 2. Token embedding + position Embedding
# 20개의 단어를 8차원 벡터로 변환
embedding = nn.Embedding(20, D)
# 4칸짜리 문장을 8차원의 벡터로 인코딩
position_embedding = nn.Embedding(T, D)

In [38]:
positions = torch.arange(T).unsqueeze(0)
positions.shape, positions

(torch.Size([1, 4]), tensor([[0, 1, 2, 3]]))

In [50]:
# 토큰 벡터들을 token_ids의 임베딩에 위치 정보까지 포함시켜주기
x = embedding(token_ids) + position_embedding(positions)
x.shape, x

(torch.Size([1, 4, 8]),
 tensor([[[-0.6214,  1.5250,  0.6353,  0.9888,  0.0552, -0.6891, -0.0768,
           -0.2341],
          [-3.0054,  1.3889,  1.2254,  0.7279, -1.6447,  2.6013, -0.4090,
            3.4101],
          [ 1.6781, -0.0390, -1.3031,  0.2320, -1.6901, -0.5126, -0.3352,
            1.1580],
          [-0.6910, -0.9775, -1.8700,  0.5541,  1.5979, -1.1975, -1.5021,
           -0.0409]]], grad_fn=<AddBackward0>))

In [40]:
# 3. Q, K, V 생성
# Query, Key, Value를 나타낼 가중치 행렬
W_q = nn.Linear(D, D, bias=False)
W_k = nn.Linear(D, D, bias=False)
W_v = nn.Linear(D, D, bias=False)

In [41]:
Q = W_q(x) # 내가 어떤 정보를 찾고싶은가
K = W_k(x) # 나는 어떤 정보를 가지고싶은가
V = W_v(x) # 실제로 내가 전달하는 정보는 무엇인가
Q.shape, K.shape, V.shape

(torch.Size([1, 4, 8]), torch.Size([1, 4, 8]), torch.Size([1, 4, 8]))

In [42]:
# Head 분리용 함수
# 원래 8차원인 벡터를 여러 Head로 쪼개어서 표현하기
# 현재 상태는 (1, 4, 8) 을 (1, 2, 4, 4)로 표현
def split_head(tensor):
    return (
        tensor
        # vector_dim=8을 2개의 head 벡터로 나누기
        .reshape(B, T, num_heads, head_dim)
        # 계산을 편하게 하기 위해 Token순서와 Head 순서를 변경
        # 이를 통해 각 Head별 독립적인 행렬곱을 한번에 수행 가능
        .transpose(1, 2)
    )

In [43]:
Q = split_head(Q)
K = split_head(K)
V = split_head(V)

Q.shape, K.shape, V.shape

(torch.Size([1, 2, 4, 4]), torch.Size([1, 2, 4, 4]), torch.Size([1, 2, 4, 4]))

In [49]:
# Attention Score
#       (4, 4) @ (4, 4) -> (4, 4)  / sqrt(d_k) (루트2)
scores = Q @ K.transpose(-2, -1) / math.sqrt(head_dim)
scores.shape

torch.Size([1, 2, 4, 4])

In [45]:
# 모든 행의 합을 1로 설정
attention = torch.softmax(scores, dim=-1)
attention.shape, attention.sum(dim=2)

(torch.Size([1, 2, 4, 4]),
 tensor([[[1.2900, 1.0251, 1.0223, 0.6625],
          [0.7906, 0.8016, 1.6834, 0.7244]]], grad_fn=<SumBackward1>))

In [46]:
# 각각 토큰들의 attention score의 비율만큼 다른 token들을 attention
# (4, 4) @ (4, 8) -> 새로운 토큰 벡터 (4, 8)
output = attention @ V

print("Output shape:", output.shape)

Output shape: torch.Size([1, 2, 4, 4])


In [47]:
# 5. Head 결합
# 차원 1과 2를 서로 바꾸고 데이터를 새로운 메모리 공간에 재배치
output = output.transpose(1, 2).contiguous()
output = output.reshape(B, T, D)

output.shape, output

(torch.Size([1, 4, 8]),
 tensor([[[ 0.1713, -0.0110,  0.0745, -0.1199,  0.6382, -0.0379, -0.0050,
           -0.2069],
          [-0.0417, -0.0979, -0.2335,  0.4357,  0.9685, -0.3514, -0.5716,
           -0.3113],
          [-0.0683, -0.1972, -0.2368,  0.4399,  0.3921, -0.4174,  0.3493,
           -0.0862],
          [ 0.0440, -0.1640, -0.0582,  0.1086,  0.2447, -0.5561,  0.5901,
           -0.0258]]], grad_fn=<ViewBackward0>))

In [48]:
# 6. Output Projection
W_o = nn.Linear(D, D) # 8, 8
output = W_o(output)

print("Final output:", output.shape)

Final output: torch.Size([1, 4, 8])
